# SOI on VBN dataset

This notebook downloads vbn.pth, instantiates datasets for each time bin and change/non_change (10 datasets), trains SB_joint (SOI) for each dataset and computes o-information, then plots results.


In [2]:
import sys
import os
import shutil
import torch
from pathlib import Path
print('Python', sys.version)
# Determine paths relative to the notebook's working directory
cwd = Path.cwd()
soi_dir = cwd
soi_repo_dir = soi_dir / 'soi_repo'
if not soi_repo_dir.exists() or not any(soi_repo_dir.iterdir()):
    print(f'soi_repo not found or empty at {soi_repo_dir}; attempting to clone...')
    if shutil.which('git') is None:
        raise RuntimeError('git is required to clone soi_repo; please install git and re-run the notebook')
    import subprocess
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/MustaphaBounoua/soi.git', str(soi_repo_dir)])
# Ensure the cloned repo is importable (add repo root so `import src...` works)
repo_root = str(soi_repo_dir)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
    print('Added repo_root to sys.path', repo_root)
os.makedirs('results/soi_vbn', exist_ok=True)
vbn_pth = os.path.join(str(soi_dir), 'vbn.pth')
if not os.path.exists(vbn_pth):
    print('vbn.pth not found at', vbn_pth)
    try:
        import gdown
    except Exception:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'gdown'])
        import gdown
    url = 'https://drive.google.com/uc?id=1HGwGm-iGVo7e573DMmp_NoFha1IaktVf'
    print('Downloading vbn.pth...')
    gdown.download(url, vbn_pth, quiet=False)
else:
    print('vbn.pth already exists at', vbn_pth)
print('Loading vbn.pth...')
vbn_bins_5 = torch.load(vbn_pth, map_location='cpu')
print('Loaded vbn.pth. Keys:', sorted(list(vbn_bins_5.keys())))


SyntaxError: invalid syntax (1234986316.py, line 4)

In [ ]:
import os
import json
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader, Dataset
from src.SB.sb_cond_vbn import SB_joint

def make_dataloader(ds, bs=128, num_workers=0):
    # Accept DataLoader, Dataset or dict-of-arrays
    if isinstance(ds, DataLoader):
        return ds
    try:
        # If it's a torch Dataset or similar
        return DataLoader(ds, batch_size=bs, shuffle=True, num_workers=num_workers, drop_last=True)
    except Exception:
        pass
    if isinstance(ds, dict):
        class DictDataset(Dataset):
            def __init__(self, data_dict):
                import torch as _t
                self.keys = list(data_dict.keys())
                self.data = {k: (_t.tensor(v) if not _t.is_tensor(v) else v) for k,v in data_dict.items()}
                self.N = next(iter(self.data.values())).shape[0]
            def __len__(self):
                return self.N
            def __getitem__(self, idx):
                return {k: self.data[k][idx] for k in self.keys}
        return DataLoader(DictDataset(ds), batch_size=bs, shuffle=True, num_workers=num_workers, drop_last=True)
    raise ValueError('Unsupported dataset type: %s' % type(ds))

def to_python(obj):
    import torch as _t
    if _t.is_tensor(obj):
        try:
            return obj.detach().cpu().item()
        except Exception:
            return obj.detach().cpu().tolist()
    if isinstance(obj, dict):
        return {k: to_python(v) for k,v in obj.items()}
    if isinstance(obj, list):
        return [to_python(v) for v in obj]
    return obj

results = []
bs = 128
epochs = 3  # small for demo; increase as needed
num_workers = 4
for time_bin in range(5):
    time_key = str(time_bin) if str(time_bin) in vbn_bins_5 else time_bin
    for change in ('change','non_change'):
        if time_key not in vbn_bins_5 or change not in vbn_bins_5[time_key]:
            print(f'Skipping dataset time_bin={time_bin}, change={change} (missing)')
            continue
        ds = vbn_bins_5[time_key][change]
        print(f'Preparing dataset time_bin={time_bin} change={change} type={type(ds)}')
        try:
            loader = make_dataloader(ds, bs=bs, num_workers=num_workers)
        except Exception as e:
            print('Failed to create dataloader:', e)
            continue
        sample = None
        for batch in loader:
            sample = batch
            break
        if sample is None:
            print('Empty dataset, skipping')
            continue
        # infer per-mod feature dim
        mod_list = {k: (v.shape[-1] if hasattr(v,'shape') and len(v.shape)>=2 else v.shape[0]) for k,v in sample.items()}
        print('mod_list:', mod_list)
        pl.seed_everything(0)
        model = SB_joint(debias=False, test_samples=loader, gt=None, lr=1e-3, mod_list=mod_list, use_ema=True, batch_size=bs, debug=False, margin_time=1, scores_order=1, fill_zeros=False, weight_subsets=True, test_epoch=epochs)
        save_dir = f'results/soi_vbn/timebin_{time_bin}_{change}'
        os.makedirs(save_dir, exist_ok=True)
        tb_logger = pl.loggers.TensorBoardLogger(save_dir, name='vbn')
        accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'
        devices = 1
        trainer = pl.Trainer(logger=tb_logger, accelerator=accelerator, devices=devices, max_epochs=epochs, enable_checkpointing=False, enable_model_summary=False)
        print(f'Training for time_bin={time_bin} change={change} for {epochs} epochs')
        trainer.fit(model, train_dataloaders=loader)
        model.eval()
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        model.to(device)
        print('Computing o-information...')
        try:
            out = model.compute_o_inf_batch(loader, debias=True, nb_iter=3)
        except Exception as e:
            print('compute_o_inf_batch failed:', e)
            out = {}
        res = {'time_bin': time_bin, 'change': change, 'result': to_python(out)}
        results.append(res)
        with open(os.path.join(save_dir, 'results.json'), 'w') as fh:
            json.dump(res, fh)
        print('Saved', save_dir)

print('Finished. Total results:', len(results))


In [ ]:
import glob, json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

files = glob.glob('results/soi_vbn/**/results.json', recursive=True)
rows = []
for f in files:
    try:
        with open(f) as fh:
            r = json.load(fh)
    except Exception as e:
        print('Failed reading', f, e)
        continue
    tb = r.get('time_bin')
    ch = r.get('change')
    metrics = r.get('result', {})
    simple = metrics.get('simple', {})
    o_inf = simple.get('o_inf')
    tc = simple.get('tc')
    dtc = simple.get('dtc')
    rows.append({'time_bin': tb, 'change': ch, 'o_inf': o_inf, 'tc': tc, 'dtc': dtc, 'file': f})
df = pd.DataFrame(rows)
if df.empty:
    print('No results found')
else:
    sns.set(style='whitegrid')
    plt.figure(figsize=(10,5))
    sns.barplot(data=df, x='time_bin', y='o_inf', hue='change')
    plt.title('O-information by time bin and condition')
    plt.show()
